In [49]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt  # for making figures
import random

%matplotlib inline

In [50]:
words = open("names.txt", "r").read().splitlines()
len(words)

32033

In [51]:
# get distinct chars across all words -- build vocab
chars = sorted(list(set("".join(words))))

# map chars to and from integers
# 0th char will be "." which denotes start/end of a word
char_to_int = {ch: idx + 1 for idx, ch in enumerate(chars)}
char_to_int["."] = 0
int_to_char = {idx: ch for ch, idx in char_to_int.items()}

print(char_to_int)
print(int_to_char)

{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}
{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [52]:
# build train, eval and test datasets
# X: 3 chars --> Y: next char
# For word "emma"
# ... --> e
# ..e --> m
# .em --> m
# emm --> a
# mma --> .

context_length = 3


def get_dataset(words):
    X, y = [], []
    for word in words:
        # start at [., ., .] case whose int rep would be [0, 0, 0]
        context = [0] * context_length
        for ch in word + ".":
            ch_idx = char_to_int[ch]
            X.append(context)
            y.append(ch_idx)
            # update context
            context = context[1:] + [ch_idx]
    X = torch.tensor(X)
    y = torch.tensor(y)
    print(X.shape, y.shape)
    return X, y


# split the dataset into train, val and test into 80-10-10 split
random.seed(42)
random.shuffle(words)

limit1 = int(0.8 * len(words))
limit2 = int(0.9 * len(words))

X_train, y_train = get_dataset(words=words[:limit1])
X_val, y_val = get_dataset(words=words[limit1:limit2])
X_test, y_test = get_dataset(words=words[limit2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [ ]:
# Initialize parameters of the network
# C --> embedding matrix. Embeds each character into an d-dimensional vector. Here, d=2
# (W1, b1) and (W2, b2) --> Two MLP layers

# Architecture
# input --> input_thro_2d_embedding --> flatten --> layer 1 (100 neurons)   --> layer 2 (27 neurons)
# (n, 3)--> (n, 3, 2)               --> (n, 6)  --> layer outputs: (n, 100) --> layer outputs (n, 27)

# Layer 2 is intentionally set to have 27 neurons to give us (n, 27) as output
# It tells what score our model gives to each of the 27 characters for a given example.
# We would want our model to give maximum score to the character next in input example's sequence.

emb_dims = 2
unique_chars = len(chars) + 1  # 27 in this case

g = torch.Generator().manual_seed(6)
C = torch.randn((unique_chars, emb_dims), generator=g, requires_grad=True)
W1 = torch.randn((6, 100), generator=g, requires_grad=True)
b1 = torch.randn(100, generator=g, requires_grad=True)
W2 = torch.randn((100, unique_chars), generator=g, requires_grad=True)
b2 = torch.randn(unique_chars, generator=g, requires_grad=True)

parameters = [C, W1, b1, W2, b2]

In [56]:
X_c = C[X_train]

In [57]:
X_c.shape

torch.Size([182625, 3, 2])

In [ ]:
X_c.view(-1, 6).shape

torch.Size([182625, 6])